[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [2]:
data_frame = pd.read_csv("2.1.2.diabeties_sample_data.csv")

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [3]:
data_frame.isnull().sum()

DoB       0
DoT       0
SEX       1
BMI       0
BP        0
TC        0
BGU       0
FDR       0
Target    1
dtype: int64

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

In [4]:
# Remove Null values
data_frame = data_frame.dropna(subset=['SEX'])
data_frame.isnull().sum()

DoB       0
DoT       0
SEX       0
BMI       0
BP        0
TC        0
BGU       0
FDR       0
Target    1
dtype: int64

In [5]:
# Replace Null values with the mean value for the column
data_frame['Target'] = data_frame['Target'].fillna(data_frame['Target'].mean())
data_frame.isnull().sum()

DoB       0
DoT       0
SEX       0
BMI       0
BP        0
TC        0
BGU       0
FDR       0
Target    0
dtype: int64

#### Remove Duplicates

Duplicate data can have detrimental effects on your machine learning models and outcomes, such as reducing data diversity and representativeness, which can lead to overfitting or biased models.

The `duplicated().sum()` method call returns the count of duplicate rows in the data frame.

In [6]:
data_frame.duplicated().sum()

np.int64(5)

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [7]:
data_frame = data_frame.drop_duplicates()
data_frame.duplicated().sum()

np.int64(0)

#### Replace data

We can run a lambda function on a column to modify its values. For a simple example, let’s convert the Sex to lowercase. To run a function over a complete column, we can use the apply method which iterates over each row and modifies the values.

In [8]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda x: x.lower())
data_frame['SEX'].head()

0    female
1    female
2      male
3      male
4      male
Name: SEX, dtype: str

We can check that there are no data entry errors by the `unique()` method call.

In [9]:
data_frame['SEX'].unique()

<StringArray>
['female', 'male', 'girl']
Length: 3, dtype: str

In [10]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda gender: 'male' if gender.lower() == 'male' else 'female')
data_frame['SEX'].unique()

<StringArray>
['female', 'male']
Length: 2, dtype: str

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [ ]:
#get the inter-quartile range on the blood pressure column
print(data_frame['TC'].describe())
Q1 = data_frame['TC'].quantile(0.25)
Q3 = data_frame['TC'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a BP above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')


In [ ]:
# Filter blood pressure within the acceptable range
data_frame = data_frame[(data_frame['BP'] >= Q1 - 1.5 * IQR) & (data_frame['BP'] <= Q3 + 1.5 * IQR)]
print(data_frame['BP'].describe())

#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [11]:
bp = 'BP'
bmi = 'BMI'
tc = 'TC'
bgu = 'BGU'

# the minimum value with space for outliers
MIN_BP = 55
MIN_BMI = 13
MIN_TC = 1
MIN_BGU = 50

# the maximum value with space for outliers
MAX_BP = 140
MAX_BMI = 48
MAX_TC = 11
MAX_BGU = 135

# scale features
data_frame[bp] = [(X - MIN_BP) / (MAX_BP - MIN_BP) for X in data_frame[bp]]
data_frame[bmi] = [(X - MIN_BMI) / (MAX_BMI - MIN_BMI) for X in data_frame[bmi]]
data_frame[tc] = [(X - MIN_TC) / (MAX_TC - MIN_TC) for X in data_frame[tc]]
data_frame[bgu] = [(X - MIN_BGU) / (MAX_BGU - MIN_BGU) for X in data_frame[bgu]]

data_frame.describe()

,BMI,BP,TC,BGU,FDR,Target
count,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000
mean,0.381357,0.466915,0.306799,0.485361,1.067873,152.030122
std,0.126326,0.167346,0.129145,0.135243,0.832944,77.198893
min,0.142857,-0.047059,0.100000,0.094118,0.000000,25.000000
25%,0.289286,0.341176,0.200000,0.391176,0.000000,86.250000
50%,0.362857,0.447059,0.300000,0.482353,1.000000,140.500000
75%,0.462857,0.588235,0.400000,0.564706,2.000000,211.500000
max,0.834286,1.011765,0.809000,0.870588,3.000000,346.000000


> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [12]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)